In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/comment-category-prediction-challenge/Sample.csv
/kaggle/input/comment-category-prediction-challenge/train.csv
/kaggle/input/comment-category-prediction-challenge/test.csv


In [2]:
import numpy as np
import pandas as pd
import gc
import re

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score

import lightgbm as lgb
from scipy.sparse import hstack, csr_matrix

In [3]:
train = pd.read_csv("/kaggle/input/comment-category-prediction-challenge/train.csv")
test = pd.read_csv("/kaggle/input/comment-category-prediction-challenge/test.csv")
sample = pd.read_csv("/kaggle/input/comment-category-prediction-challenge/Sample.csv")

train.columns = train.columns.str.strip()
test.columns = test.columns.str.strip()


In [4]:
drop_cols = ["race", "religion", "gender", "created_date"]

train = train.drop(columns=drop_cols)
test = test.drop(columns=drop_cols)

train["comment"] = train["comment"].fillna("")
test["comment"] = test["comment"].fillna("")

In [5]:
def add_features(df):
    df["char_count"] = df["comment"].apply(len)
    df["word_count"] = df["comment"].apply(lambda x: len(x.split()))
    df["unique_word_count"] = df["comment"].apply(lambda x: len(set(x.split())))

    df["upper_count"] = df["comment"].apply(lambda x: sum(1 for c in x if c.isupper()))
    df["digit_count"] = df["comment"].apply(lambda x: sum(1 for c in x if c.isdigit()))
    df["special_count"] = df["comment"].apply(lambda x: len(re.findall(r'[!?.]', x)))

    
    df["upper_ratio"] = df["upper_count"] / (df["char_count"] + 1)
    df["unique_ratio"] = df["unique_word_count"] / (df["word_count"] + 1)
    df["special_ratio"] = df["special_count"] / (df["char_count"] + 1)

    
    df["avg_word_len"] = df["char_count"] / (df["word_count"] + 1)
    df["caps_per_word"] = df["upper_count"] / (df["word_count"] + 1)

    return df

train = add_features(train)
test = add_features(test)

In [6]:
X = train.drop("label", axis=1)
y = train["label"]

num_cols = [col for col in X.columns if col != "comment"]


In [7]:
X_num = X[num_cols].apply(pd.to_numeric, errors="coerce").fillna(0).astype(float).values
test_num = test[num_cols].apply(pd.to_numeric, errors="coerce").fillna(0).astype(float).values

X_num = csr_matrix(X_num)
test_num = csr_matrix(test_num)

In [8]:
tfidf_word = TfidfVectorizer(
    max_features=6000,
    ngram_range=(1,2),
    min_df=2
)

tfidf_char = TfidfVectorizer(
    analyzer="char",
    ngram_range=(3,5),
    max_features=3000
)

X_word = tfidf_word.fit_transform(X["comment"])
X_char = tfidf_char.fit_transform(X["comment"])

test_word = tfidf_word.transform(test["comment"])
test_char = tfidf_char.transform(test["comment"])

X_text = hstack([X_word, X_char])
test_text = hstack([test_word, test_char])


In [9]:
X_all = hstack([X_num, X_text])
test_all = hstack([test_num, test_text])


In [10]:
X_train, X_val, y_train, y_val = train_test_split(
    X_all, y, test_size=0.2, random_state=42, stratify=y
)

In [11]:
model = lgb.LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.045,
    num_leaves=40,
    subsample=0.7,
    colsample_bytree=0.7,
    random_state=42
)

In [12]:
model.fit(X_train, y_train)

pred = model.predict(X_val)
print("Validation F1:", f1_score(y_val, pred, average="macro"))

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 16.076440 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1750759
[LightGBM] [Info] Number of data points in the train set: 158400, number of used features: 9020
[LightGBM] [Info] Start training from score -0.550557
[LightGBM] [Info] Start training from score -2.520769
[LightGBM] [Info] Start training from score -1.154061
[LightGBM] [Info] Start training from score -3.589217


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Validation F1: 0.8090947069690627


In [13]:
model.fit(X_all, y)

test_pred = model.predict(test_all)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 23.886011 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1877264
[LightGBM] [Info] Number of data points in the train set: 198000, number of used features: 9020
[LightGBM] [Info] Start training from score -0.550552
[LightGBM] [Info] Start training from score -2.520816
[LightGBM] [Info] Start training from score -1.154061
[LightGBM] [Info] Start training from score -3.589171


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [14]:
submission = sample.copy()
submission["label"] = test_pred

submission.to_csv("submission.csv", index=False)

submission.head()

,ID,label
0,1,2
1,2,2
2,3,0
3,4,0
4,5,2
